In [ ]:
# General imports
### test
### test 0410
### for 
### 1111111111111111111
### 22222222222222222222
### 33333333333333333333
import numpy as np
import pandas as pd
import os, sys, gc, time, warnings, pickle, psutil, random
from math import ceil
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

warnings.filterwarnings('ignore')

In [ ]:
#################################################################################
# Some preparation
#################################################################################
# MAIN_INDEX = ['id','d']  # the link between different dataset
TARGET = 'sales'         # Our main target
END_TRAIN = 1941         # Last day in train set
# load data
train_df = pd.read_csv("sales_train_evaluation.csv")
prices_df = pd.read_csv("sell_prices.csv")
calendar_df = pd.read_csv("calendar.csv")

## Merging by concat to preserve data types
def merge_by_concat(df1, df2, merge_on):
    """
    Merges two DataFrames by concatenation, preserving data types in the original DataFrame.
    
    Parameters:
        df1 (pd.DataFrame): The main DataFrame to which data will be merged.
        df2 (pd.DataFrame): The secondary DataFrame to merge data from.
        merge_on (list): The column(s) to use as the key(s) for the merge.
        
    Returns:
        pd.DataFrame: The resulting DataFrame after merging.
    """
    # Perform a left merge between the DataFrames on the specified keys
    merged_temp = df1[merge_on].merge(df2, on=merge_on, how='left')
    
    # Identify the new columns added from the second DataFrame
    new_columns = [col for col in merged_temp.columns if col not in merge_on]
    
    # Concatenate the new columns to the original DataFrame
    df1 = pd.concat([df1, merged_temp[new_columns]], axis=1)
    
    return df1

def reduce_mem_usage(df, verbose=True):
    """
    Reduces memory usage of a Pandas DataFrame by downcasting numeric columns.

    Parameters:
        df (pd.DataFrame): The DataFrame to optimize.
        verbose (bool): Whether to print memory usage details.

    Returns:
        pd.DataFrame: The optimized DataFrame with reduced memory usage.
    """
    # Define numeric data types to check for optimization
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    
    # Calculate initial memory usage
    start_mem = df.memory_usage().sum() / 1024**2

    # Iterate through all columns in the DataFrame
    for col in df.columns:
        col_type = df[col].dtypes

        # Optimize numeric columns only
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()

            # Downcast integers
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)

            # Downcast floats
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    
    # Calculate final memory usage
    end_mem = df.memory_usage().sum() / 1024**2

    # Print memory reduction details if verbose is enabled
    if verbose:
        print(f'Memory usage reduced to {end_mem:5.2f} MB ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    
    return df

In [ ]:
#################################################################################
# Make long panel data
#################################################################################
# Transform horizontal representation into a vertical view
# Our index columns are 'id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'
# The label columns are the 'd_' columns
index_columns = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

# Reshape the training data from wide format to long format
grid_df = pd.melt(
    train_df,
    id_vars=index_columns,
    var_name='d',
    value_name=TARGET
)

# Print the number of rows in the original training set and the reshaped grid
print('Train rows:', len(train_df), len(grid_df))

# To enable predictions, we need to add a test set to the grid
test_days = 28
add_grid = []

# Generate test set data for each day after the training period
for i in range(1, test_days + 1):
    # Extract unique rows for the index columns
    temp_df = train_df[index_columns].drop_duplicates()
    # Create a new 'd' column for the current test day
    temp_df['d'] = f'd_{END_TRAIN + i}'
    # Assign NaN to the TARGET column for the test set
    temp_df[TARGET] = np.nan
    add_grid.append(temp_df)

# Combine all test set data into a single DataFrame
add_grid = pd.concat(add_grid, ignore_index=True)
# Append the test set data to the original grid
grid_df = pd.concat([grid_df, add_grid], ignore_index=True)

# Convert index columns to the 'category' type to save memory
grid_df[index_columns] = grid_df[index_columns].astype('category')

# Reset the index of the final grid
grid_df = grid_df.reset_index(drop=True)

#################################################################################

Create Grid
Train rows: 30490 59181090


In [ ]:
#################################################################################
# Process Calender dataset ( del the data that we do not need)
#################################################################################

# Leading zero values in `train_df` rows are likely not real sales values
# but indicate the absence of the item in the store.
# Removing such zeros can save memory.

# Prices are organized by week, so the release week may not be very precise.
# Calculate the earliest release week for each product in each store.
release_df = prices_df.groupby(['store_id', 'item_id'])['wm_yr_wk'].min().reset_index()
release_df.columns = ['store_id', 'item_id', 'release']

# Merge the release data into `grid_df`.
grid_df = merge_by_concat(grid_df, release_df, ['store_id', 'item_id'])
del release_df  # Free up memory

# To remove unnecessary rows (e.g., early zero sales),
# we need to merge the `wm_yr_wk` column from the calendar data.
grid_df = merge_by_concat(grid_df, calendar_df[['wm_yr_wk', 'd']], ['d'])

# Filter out rows where the week is earlier than the product release week.
grid_df = grid_df[grid_df['wm_yr_wk'] >= grid_df['release']]
grid_df = grid_df.reset_index(drop=True)

# Check memory usage before optimization
print("{:>20}: {:>8}".format('Original grid_df', sizeof_fmt(grid_df.memory_usage(index=True).sum())))

# Transform the release week to reduce memory usage:
# Offset the release values by subtracting the minimum release week.
# Convert to `int16` to save space while retaining precision.
grid_df['release'] = grid_df['release'] - grid_df['release'].min()
grid_df['release'] = grid_df['release'].astype(np.int16)

# Check memory usage after optimization
print("{:>20}: {:>8}".format('Reduced grid_df', sizeof_fmt(grid_df.memory_usage(index=True).sum())))

In [ ]:
#################################################################################
# Save Base Grid
#################################################################################

# The base grid is ready and can be saved for future use,
# such as model training. Using pickle allows for efficient
# storage and loading of DataFrame objects.
output_file = 'grid_part_1.pkl'
# Save the DataFrame to a pickle file
grid_df.to_pickle(output_file)

Save Part 1
Size: (47735397, 10)


In [ ]:
#################################################################################
# Prices
#################################################################################

# 1. Calculate basic statistical values for item prices in each store
# Max, min, standard deviation, and mean prices for each item in each store
prices_df['price_max'] = prices_df.groupby(['store_id', 'item_id'])['sell_price'].transform('max')
prices_df['price_min'] = prices_df.groupby(['store_id', 'item_id'])['sell_price'].transform('min')
prices_df['price_std'] = prices_df.groupby(['store_id', 'item_id'])['sell_price'].transform('std')
prices_df['price_mean'] = prices_df.groupby(['store_id', 'item_id'])['sell_price'].transform('mean')

# 2. Normalize item prices to the range [0, 1] using min-max scaling
prices_df['price_norm'] = prices_df['sell_price'] / prices_df['price_max']

# 3. Calculate additional metrics:
# - `price_nunique`: Count of unique prices for each item in each store.
# - `item_nunique`: Count of unique items sold at a given price point in each store.
# Purpose:
# - `price_nunique`: Indicates how frequently an item's price changes.
# - `item_nunique`: Analyzes whether a price point corresponds to multiple items, reflecting the price's generality.
prices_df['price_nunique'] = prices_df.groupby(['store_id', 'item_id'])['sell_price'].transform('nunique')
prices_df['item_nunique'] = prices_df.groupby(['store_id', 'sell_price'])['item_id'].transform('nunique')

# 4. Merge date information (month and year) from `calendar_df` into `prices_df`
# This enables time-based aggregations for rolling calculations.
calendar_prices = calendar_df[['wm_yr_wk', 'month', 'year']].drop_duplicates(subset=['wm_yr_wk'])
prices_df = prices_df.merge(calendar_prices[['wm_yr_wk', 'month', 'year']], on='wm_yr_wk', how='left')
del calendar_prices  # Free up memory

# 5. Add price momentum metrics:
# - `price_momentum`: Week-over-week price ratio (current price vs. previous week).
# - `price_momentum_m`: Price ratio relative to the monthly average price.
# - `price_momentum_y`: Price ratio relative to the yearly average price.

# Week-over-week price ratio
prices_df['price_momentum'] = prices_df['sell_price'] / prices_df.groupby(['store_id', 'item_id'])['sell_price'].transform(lambda x: x.shift(1))

# Price ratio relative to monthly mean price
prices_df['price_momentum_m'] = prices_df['sell_price'] / prices_df.groupby(['store_id', 'item_id', 'month'])['sell_price'].transform('mean')

# Price ratio relative to yearly mean price
prices_df['price_momentum_y'] = prices_df['sell_price'] / prices_df.groupby(['store_id', 'item_id', 'year'])['sell_price'].transform('mean')

# Remove temporary columns `month` and `year` after calculations
del prices_df['month'], prices_df['year']

grid_df = reduce_mem_usage(grid_df)
prices_df = reduce_mem_usage(prices_df)

Prices


In [ ]:
#################################################################################
# Merge Prices and Save Part 2
#################################################################################
# 1. Merge `prices_df` into `grid_df` on the keys: 'store_id', 'item_id', 'wm_yr_wk'
# This adds all price-related features to the main grid
original_columns = list(grid_df.columns)
grid_df = grid_df.merge(prices_df, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')

# 2. Identify the newly added columns (price-related features)
keep_columns = [col for col in grid_df.columns if col not in original_columns]

# 3. Retain only the main index columns (`MAIN_INDEX`) and the newly added price-related columns
grid_df = grid_df[MAIN_INDEX + keep_columns]

# 4. Optimize memory usage of `grid_df`
grid_df = reduce_mem_usage(grid_df)

# 5. Save the updated grid to a pickle file for future use
output_file = 'grid_part_2.pkl'
grid_df.to_pickle(output_file)

# 6. Print the size of the final DataFrame for verification
print(f'Size: {grid_df.shape}')

Merge prices and save part 2
Mem. usage decreased to 1412.49 Mb (0.0% reduction)
Size: (47735397, 13)


In [ ]:
#################################################################################
# Process Calendar Information and Save Part 3 (Dates)
#################################################################################

# Remove unused DataFrames to free up memory
del prices_df, grid_df

# Reload the base grid from part_1 and retain only the main index columns
grid_df = pd.read_pickle('grid_part_1.pkl')
grid_df = grid_df[MAIN_INDEX]

# 1. Merge date-related information from `calendar_df`
date_columns = [
    'date',
    'd',
    'event_name_1',
    'event_type_1',
    'event_name_2',
    'event_type_2',
    'snap_CA',
    'snap_TX',
    'snap_WI'
]

grid_df = grid_df.merge(calendar_df[date_columns], on='d', how='left')

# - `date`: Actual date.
# - `event_name_1`, `event_name_2`: Event names (e.g., holiday names).
# - `event_type_1`, `event_type_2`: Event types (e.g., Religious, Cultural).
# - `snap_CA`, `snap_TX`, `snap_WI`: Indicates if SNAP was active in California, Texas, or Wisconsin.

# 2. Convert certain columns to `category` for memory efficiency
categorical_columns = [
    'event_name_1',
    'event_type_1',
    'event_name_2',
    'event_type_2',
    'snap_CA',
    'snap_TX',
    'snap_WI'
]

for col in categorical_columns:
    grid_df[col] = grid_df[col].astype('category')

# Convert `date` column to datetime format
grid_df['date'] = pd.to_datetime(grid_df['date'])

# 3. Extract date-related features
# - `tm_d`: Day of the month.
# - `tm_w`: Week of the year.
# - `tm_m`: Month of the year.
# - `tm_y`: Relative year (current year - minimum year, to minimize values).
# - `tm_wm`: Week of the month (ceil(day_of_month / 7)).
# - `tm_dw`: Day of the week (0=Monday, 6=Sunday).
# - `tm_w_end`: Is weekend (1 if tm_dw >= 5, otherwise 0).

grid_df['tm_d'] = grid_df['date'].dt.day.astype(np.int8)
grid_df['tm_w'] = grid_df['date'].dt.isocalendar().week.astype(np.int8)  # ISO week number
grid_df['tm_m'] = grid_df['date'].dt.month.astype(np.int8)
grid_df['tm_y'] = grid_df['date'].dt.year
grid_df['tm_y'] = (grid_df['tm_y'] - grid_df['tm_y'].min()).astype(np.int8)
grid_df['tm_wm'] = grid_df['tm_d'].apply(lambda x: ceil(x / 7)).astype(np.int8)
grid_df['tm_dw'] = grid_df['date'].dt.dayofweek.astype(np.int8)
grid_df['tm_w_end'] = (grid_df['tm_dw'] >= 5).astype(np.int8)

# Remove the `date` column after extracting features
del grid_df['date']

#################################################################################
# Save Part 3 (Dates)
#################################################################################

print('Save part 3')

# Save the updated grid to a pickle file
output_file = 'grid_part_3.pkl'
grid_df.to_pickle(output_file)

# Print the size of the DataFrame for verification
print(f'Size: {grid_df.shape}')

# Remove unused DataFrames to free up memory
del calendar_df
del grid_df

In [ ]:
#################################################################################
# Generate Lags and Rollings
#################################################################################

# Load the base grid from part 1
grid_df = pd.read_pickle('grid_part_1.pkl')

# Retain only the necessary columns: 'id', 'd', 'sales'
# This reduces memory usage and simplifies computations.
grid_df = grid_df[['id', 'd', 'sales']]
SHIFT_DAY = 28  # Define the shift period for lag and rolling calculations

#################################################################################
# Lags
#################################################################################

print('Create lags')
start_time = time.time()

# Define the lag days to compute (e.g., 28 to 42)
LAG_DAYS = range(SHIFT_DAY, SHIFT_DAY + 15)

# Create lagged features for each specified lag day
grid_df = grid_df.assign(**{
    f'{TARGET}_lag_{lag}': grid_df.groupby(['id'])[TARGET].transform(lambda x: x.shift(lag))
    for lag in LAG_DAYS
})

# Minify lag columns by converting to `float16` to save memory
for col in grid_df.columns:
    if 'lag' in col:
        grid_df[col] = grid_df[col].astype(np.float16)

print('%.2f min: Lags' % ((time.time() - start_time) / 60))

#################################################################################
# Rollings with Fixed Shift
#################################################################################

print('Create rolling aggs')
start_time = time.time()

# Calculate rolling mean and standard deviation for various window sizes
for window in [7, 14, 30, 60, 180]:
    print('Rolling period:', window)
    grid_df[f'rolling_mean_{window}'] = grid_df.groupby(['id'])[TARGET] \
        .transform(lambda x: x.shift(SHIFT_DAY).rolling(window).mean()).astype(np.float16)
    grid_df[f'rolling_std_{window}'] = grid_df.groupby(['id'])[TARGET] \
        .transform(lambda x: x.shift(SHIFT_DAY).rolling(window).std()).astype(np.float16)

#################################################################################
# Rollings with Sliding Shift
#################################################################################

# Calculate rolling mean with different sliding shifts and window sizes
for shift in [1, 7, 14]:
    print('Shifting period:', shift)
    for window in [7, 14, 30, 60]:
        col_name = f'rolling_mean_tmp_{shift}_{window}'
        grid_df[col_name] = grid_df.groupby(['id'])[TARGET] \
            .transform(lambda x: x.shift(shift).rolling(window).mean()).astype(np.float16)

print('%.2f min: Rollings' % ((time.time() - start_time) / 60))

#################################################################################
# Export Lags and Rollings
#################################################################################

print('Save lags and rollings')
output_file = f'lags_df_{SHIFT_DAY}.pkl'
grid_df.to_pickle(output_file)
print(f'Lags and rollings saved to {output_file}')

Create lags
2.96 min: Lags
Create rolling aggs
Rolling period: 7
Rolling period: 14
Rolling period: 30
Rolling period: 60
Rolling period: 180
Shifting period: 1
Shifting period: 7
Shifting period: 14
5.13 min: Lags


In [ ]:
#################################################################################
# Generate Mean/Std features
#################################################################################

# Load the base grid to ensure alignment by index
grid_df = pd.read_pickle('grid_part_1.pkl')

# Set the last 28 days' `sales` values to NaN for encoding purposes
# This ensures we do not use future data for encoding
grid_df.loc[grid_df['d'] > (1941 - 28), 'sales'] = np.nan

# Save the original columns for later reference
base_cols = list(grid_df.columns)

# Define the column groups for which mean and standard deviation encoding will be calculated
icols = [
    ['state_id'],
    ['store_id'],
    ['cat_id'],
    ['dept_id'],
    ['state_id', 'cat_id'],
    ['state_id', 'dept_id'],
    ['store_id', 'cat_id'],
    ['store_id', 'dept_id'],
    ['item_id'],
    ['item_id', 'state_id'],
    ['item_id', 'store_id']
]

# Calculate mean and standard deviation for each column group
for col_group in icols:
    print('Encoding', col_group)
    col_name = '_' + '_'.join(col_group) + '_'
    
    # Compute group-wise mean and standard deviation for `sales`
    grid_df[f'enc{col_name}mean'] = grid_df.groupby(col_group)['sales'].transform('mean').astype(np.float16)
    grid_df[f'enc{col_name}std'] = grid_df.groupby(col_group)['sales'].transform('std').astype(np.float16)

# Identify the new columns (mean/std encoding features)
keep_cols = [col for col in grid_df.columns if col not in base_cols]

# Retain only the main index (`id`, `d`) and the new encoding features
grid_df = grid_df[['id', 'd'] + keep_cols]

#################################################################################
# Save the Mean/Std Encoding Features
#################################################################################

output_file = 'mean_encoding_df.pkl'
print('Save Mean/Std encoding')
grid_df.to_pickle(output_file)

print(f'Mean/Std encoding saved to {output_file}')

Encoding ['state_id']
Encoding ['store_id']
Encoding ['cat_id']
Encoding ['dept_id']
Encoding ['state_id', 'cat_id']
Encoding ['state_id', 'dept_id']
Encoding ['store_id', 'cat_id']
Encoding ['store_id', 'dept_id']
Encoding ['item_id']
Encoding ['item_id', 'state_id']
Encoding ['item_id', 'store_id']
